# Setup

In [1]:
import duckdb
from splink import Linker, DuckDBAPI, SettingsCreator, block_on
import splink.comparison_library as cl
import splink.comparison_level_library as cll

con = duckdb.connect()

df_l = con.read_parquet("./perturbed_data_l.parquet")
df_r = con.read_parquet("./perturbed_data_r.parquet")

db_api = DuckDBAPI(con)

df_l = db_api.register(df_l, table_name="df_l")
df_r = db_api.register(df_r, table_name="df_r")

# Predictions

In [ ]:
# reuse the fitted model
linker = Linker(
    splink_dataframe_or_dataframes=[df_l, df_r],
    settings="./model.json",
    log_level=20,
)

df_predict = linker.inference.predict(
    threshold_match_probability=0.1,
)

In [ ]:
con.sql(f"select count(*) from {df_predict.physical_name}")

# Clustering

In [ ]:
df_clusters = linker.clustering.cluster_pairwise_predictions_at_threshold(
    df_predict=df_predict,
    threshold_match_probability=0.9,
)

# df_clusters = linker.clustering.cluster_using_single_best_links(
#     df_predict=df_predict,
#     threshold_match_probability=0.8,
#     duplicate_free_datasets=['df_l', 'df_r'],
# )

In [ ]:
# investigate cluster sizes
cluster_sizes = con.sql(
    f"""
    select
        cluster_id,
        sum(case when source_dataset = 'df_l' then 1 else 0 end) as size_l,
        sum(case when source_dataset = 'df_r' then 1 else 0 end) as size_r
    from {df_clusters.physical_name}
    group by all
    """
)

con.sql(
    f"""
    select
        size_l,
        size_r,
        count(*) as count
    from cluster_sizes
    group by all
    order by count desc
    """
)

In [ ]:
# investigate person_ids which are split across two or more clusters (missed links)
splits = con.sql(
    f"""
    select
        person_id,
        count(distinct cluster_id) > 1 as split
    from {df_clusters.physical_name}
    group by person_id
    """
)

con.sql(
    f"""
    select 
        split,
        count(*) as count
    from splits
    group by split
    """
)

In [ ]:
# investigate clusters with more than one person id (false links)
merges = con.sql(
    f"""
    select
        cluster_id,
        count(distinct person_id) > 1 as merged
    from {df_clusters.physical_name}
    group by cluster_id
    """
)

con.sql(
    f"""
    select 
        merged,
        count(*) as count
    from merges
    group by merged
    """
)